<a href="https://colab.research.google.com/github/arnavon2005/Army_Provost_ML_Project/blob/main/07_Army_Provost_Dashboard.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Army Provost Decision Support Dashboard — Version 1.0

## Control Room — Applying Machine Learning for Indian Army Provost Functioning

This notebook contains the final dashboard integration, deployment workflow, and validation framework for the Army Provost Machine Learning Decision Support System.

The dashboard integrates:

- Random Forest arrest-likelihood prediction
- Army Provost incident taxonomy
- Operational response mapping
- Structured response guidance
- Human-in-the-loop decision support
- Session-level recent decision history
- Persistent DSS audit logging
- Streamlit dashboard interface
- Cloudflare Quick Tunnel deployment

> **System Classification:** Engineering decision-support prototype.  
> It is not an autonomous command system and does not represent official Indian Army operational SOP.

In [2]:
# ============================================================
# ARMY PROVOST DASHBOARD
# SECTION 1 — ENVIRONMENT & BACKEND CONNECTION
# ============================================================

import os
import sys
import importlib
from datetime import datetime

# ------------------------------------------------------------
# Google Drive
# ------------------------------------------------------------

from google.colab import drive

drive.mount("/content/drive")

# ------------------------------------------------------------
# Project paths
# ------------------------------------------------------------

PROJECT_ROOT = (
    "/content/drive/MyDrive/Army_Provost_ML_Project"
)

OUTPUTS_PATH = os.path.join(
    PROJECT_ROOT,
    "Outputs"
)

MODELS_PATH = os.path.join(
    PROJECT_ROOT,
    "Models"
)

BACKEND_PATH = os.path.join(
    OUTPUTS_PATH,
    "army_provost_dss_backend.py"
)

# ------------------------------------------------------------
# Environment verification
# ------------------------------------------------------------

print("=" * 80)
print("ARMY PROVOST DASHBOARD — ENVIRONMENT CHECK")
print("=" * 80)

print(f"Project root : {PROJECT_ROOT}")
print(f"Outputs path : {OUTPUTS_PATH}")
print(f"Models path  : {MODELS_PATH}")
print(f"Backend path : {BACKEND_PATH}")

print("\nPath verification:")

paths_to_check = {
    "Project root": PROJECT_ROOT,
    "Outputs": OUTPUTS_PATH,
    "Models": MODELS_PATH,
    "DSS backend": BACKEND_PATH
}

for name, path in paths_to_check.items():

    exists = os.path.exists(path)

    print(
        f"  {'✅' if exists else '❌'} "
        f"{name}: {exists}"
    )

# ------------------------------------------------------------
# Add persistent backend location to Python path
# ------------------------------------------------------------

if OUTPUTS_PATH not in sys.path:
    sys.path.insert(0, OUTPUTS_PATH)

# ------------------------------------------------------------
# Import persistent DSS backend
# ------------------------------------------------------------

backend = importlib.import_module(
    "army_provost_dss_backend"
)

print("\n" + "=" * 80)
print("DSS BACKEND CONNECTION")
print("=" * 80)

print("✅ Persistent DSS backend imported successfully.")

# ------------------------------------------------------------
# Verify dashboard-facing function
# ------------------------------------------------------------

if not hasattr(
    backend,
    "execute_dashboard_dss"
):

    raise ImportError(
        "execute_dashboard_dss() was not found "
        "in the persistent DSS backend."
    )

execute_dashboard_dss = (
    backend.execute_dashboard_dss
)

print(
    "✅ execute_dashboard_dss() "
    "is available."
)

# ------------------------------------------------------------
# Backend metadata
# ------------------------------------------------------------

print("\n" + "=" * 80)
print("DASHBOARD BACKEND READY")
print("=" * 80)

print(f"Backend module : {backend.__name__}")
print(
    f"Backend file   : "
    f"{getattr(backend, '__file__', 'Unknown')}"
)

print(
    f"Verified at    : "
    f"{datetime.now().strftime('%Y-%m-%d %H:%M:%S')}"
)

print("\n" + "=" * 80)
print("✅ SECTION 1 COMPLETE")
print("=" * 80)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
ARMY PROVOST DASHBOARD — ENVIRONMENT CHECK
Project root : /content/drive/MyDrive/Army_Provost_ML_Project
Outputs path : /content/drive/MyDrive/Army_Provost_ML_Project/Outputs
Models path  : /content/drive/MyDrive/Army_Provost_ML_Project/Models
Backend path : /content/drive/MyDrive/Army_Provost_ML_Project/Outputs/army_provost_dss_backend.py

Path verification:
  ✅ Project root: True
  ✅ Outputs: True
  ✅ Models: True
  ✅ DSS backend: True

DSS BACKEND CONNECTION
✅ Persistent DSS backend imported successfully.
✅ execute_dashboard_dss() is available.

DASHBOARD BACKEND READY
Backend module : army_provost_dss_backend
Backend file   : /content/drive/MyDrive/Army_Provost_ML_Project/Outputs/army_provost_dss_backend.py
Verified at    : 2026-08-09 17:19:44

✅ SECTION 1 COMPLETE


## Section 2 — DSS Backend Integration Test

This section verifies that the dashboard notebook can successfully communicate with the persistent Army Provost DSS backend.

A representative BATTERY incident is used as the integration test case.

The underlying model and DSS logic are not modified.

In [3]:
# ============================================================
# ARMY PROVOST DASHBOARD
# SECTION 2 — DSS BACKEND INTEGRATION TEST
# ============================================================

print("=" * 80)
print("ARMY PROVOST DASHBOARD — DSS INTEGRATION TEST")
print("=" * 80)

# ------------------------------------------------------------
# Representative operator input
# ------------------------------------------------------------

test_input = {
    "primary_type": "BATTERY",
    "location_description": "STREET",
    "domestic": False,
    "year": 2025,
    "month": 7,
    "day": 15,
    "hour": 20,
    "district": 1,
    "beat": 100,
    "ward": 1,
    "community_area": 32
}

print("\nTest incident:")
for key, value in test_input.items():
    print(f"  {key}: {value}")

# ------------------------------------------------------------
# Execute persistent DSS backend
# ------------------------------------------------------------

dashboard_result = execute_dashboard_dss(
    **test_input
)

# ------------------------------------------------------------
# Verify result structure
# ------------------------------------------------------------

required_top_level_keys = [
    "Validated Input",
    "DSS Result"
]

missing_top_level_keys = [
    key
    for key in required_top_level_keys
    if key not in dashboard_result
]

if missing_top_level_keys:
    raise RuntimeError(
        f"Missing top-level DSS keys: "
        f"{missing_top_level_keys}"
    )

dss_result = dashboard_result["DSS Result"]

required_dss_keys = [
    "Decision",
    "Decision Record",
    "Interpretation",
    "Operator Summary"
]

missing_dss_keys = [
    key
    for key in required_dss_keys
    if key not in dss_result
]

if missing_dss_keys:
    raise RuntimeError(
        f"Missing DSS result keys: "
        f"{missing_dss_keys}"
    )

decision = dss_result["Decision"]

# ------------------------------------------------------------
# Display integration result
# ------------------------------------------------------------

print("\n" + "=" * 80)
print("DSS RESULT")
print("=" * 80)

print(
    f"Provost Category : "
    f"{decision['Provost Incident Category']}"
)

print(
    f"Subcategory      : "
    f"{decision['Incident Subcategory']}"
)

print(
    f"Priority         : "
    f"{decision['Priority']}"
)

print(
    f"Arrest Probability: "
    f"{decision['Arrest Probability (%)']}%"
)

print(
    f"Arrest Prediction : "
    f"{decision['Arrest Prediction']}"
)

print(
    f"Response Pathway  : "
    f"{decision['Recommended Response Type']}"
)

print(
    f"Decision ID       : "
    f"{dss_result['Decision Record']['Decision ID']}"
)

# ------------------------------------------------------------
# Integration verification
# ------------------------------------------------------------

assert (
    decision["Primary Type"] == "BATTERY"
)

assert (
    decision["Provost Incident Category"]
    == "Personnel & Physical Safety"
)

assert (
    decision["Incident Subcategory"]
    == "Physical Assault"
)

assert (
    decision["Priority"]
    == "High"
)

assert (
    decision["Arrest Probability (%)"]
    == 39.57
)

assert (
    decision["Arrest Prediction"]
    == "Less Likely Arrest"
)

assert (
    decision["Recommended Response Type"]
    == "Personnel Safety / Provost Response"
)

print("\n" + "=" * 80)
print("✅ DSS BACKEND INTEGRATION TEST PASSED")
print("=" * 80)

ARMY PROVOST DASHBOARD — DSS INTEGRATION TEST

Test incident:
  primary_type: BATTERY
  location_description: STREET
  domestic: False
  year: 2025
  month: 7
  day: 15
  hour: 20
  district: 1
  beat: 100
  ward: 1
  community_area: 32

DSS RESULT
Provost Category : Personnel & Physical Safety
Subcategory      : Physical Assault
Priority         : High
Arrest Probability: 39.57%
Arrest Prediction : Less Likely Arrest
Response Pathway  : Personnel Safety / Provost Response
Decision ID       : AP-DSS-20260809-171944-501188

✅ DSS BACKEND INTEGRATION TEST PASSED


# Section 3 — Operator Input Interface

The dashboard provides an operator-facing interface for entering incident information.

This section defines the input controls only.

The inputs will later be passed to the existing `execute_dashboard_dss()` function from the persistent Army Provost DSS backend.

No machine-learning model or DSS decision logic is implemented in this section.

In [4]:
# ============================================================
# ARMY PROVOST DASHBOARD
# SECTION 3 — STREAMLIT DEPENDENCY SETUP
# ============================================================

import sys
import subprocess

print("=" * 80)
print("ARMY PROVOST DASHBOARD — STREAMLIT SETUP")
print("=" * 80)

# Install Streamlit only if it is not already available
try:
    import streamlit as st

    print("✅ Streamlit is already installed.")
    print(f"Version: {st.__version__}")

except ImportError:

    print("Streamlit not found.")
    print("Installing Streamlit...")

    subprocess.check_call([
        sys.executable,
        "-m",
        "pip",
        "install",
        "-q",
        "streamlit"
    ])

    import streamlit as st

    print("✅ Streamlit installed successfully.")
    print(f"Version: {st.__version__}")

print("=" * 80)
print("✅ STREAMLIT DEPENDENCY READY")
print("=" * 80)

ARMY PROVOST DASHBOARD — STREAMLIT SETUP
✅ Streamlit is already installed.
Version: 1.61.1
✅ STREAMLIT DEPENDENCY READY


# Section 3 — Dashboard Application

The Streamlit dashboard is maintained as a standalone Python application in the
existing `Outputs` directory.

This notebook generates and updates the application file.

The dashboard uses the persistent Army Provost DSS backend and does not duplicate
or modify the machine-learning model or DSS decision logic.

In [5]:
# ============================================================
# ARMY PROVOST DASHBOARD
# MASTER DASHBOARD APPLICATION GENERATOR
#
# FEATURES:
# - Taxonomy-driven Incident Type dropdown
# - Preprocessor-driven Location Description dropdown
# - Stable HTML/CSS rendering through st.html()
# - ML arrest-probability interpretation clarification
# - Structured Response Guidance integration
# - Session-level Recent Decision History using st.session_state
# - Persistent DSS audit logging to the existing Logs/ directory
# - Existing validated DSS/backend preserved
# ============================================================

import os


# ------------------------------------------------------------
# Dashboard application path
# ------------------------------------------------------------

DASHBOARD_APP_PATH = os.path.join(
    OUTPUTS_PATH,
    "army_provost_dashboard.py"
)


# ------------------------------------------------------------
# Complete Streamlit dashboard source
# ------------------------------------------------------------

dashboard_code = r'''
import csv
import os
import sys

import joblib
import streamlit as st


# ============================================================
# SESSION-LEVEL DECISION HISTORY
# ============================================================

if "recent_decisions" not in st.session_state:

    st.session_state.recent_decisions = []


MAX_RECENT_DECISIONS = 20


# ============================================================
# ARMY PROVOST DECISION SUPPORT SYSTEM
# ============================================================


# ------------------------------------------------------------
# Permanent project paths
# ------------------------------------------------------------

PROJECT_ROOT = (
    "/content/drive/MyDrive/Army_Provost_ML_Project"
)

OUTPUTS_PATH = os.path.join(
    PROJECT_ROOT,
    "Outputs"
)

MODELS_PATH = os.path.join(
    PROJECT_ROOT,
    "Models"
)

LOGS_PATH = os.path.join(
    PROJECT_ROOT,
    "Logs"
)

AUDIT_LOG_PATH = os.path.join(
    LOGS_PATH,
    "Army_Provost_DSS_Audit_Log.csv"
)

BACKEND_PATH = os.path.join(
    OUTPUTS_PATH,
    "army_provost_dss_backend.py"
)

PREPROCESSOR_PATH = os.path.join(
    MODELS_PATH,
    "preprocessing_pipeline.pkl"
)


# ------------------------------------------------------------
# Required artifact validation
# ------------------------------------------------------------

if not os.path.exists(BACKEND_PATH):

    st.error(
        "Army Provost DSS backend could not be found."
    )

    st.stop()


if not os.path.exists(PREPROCESSOR_PATH):

    st.error(
        "Army Provost preprocessing pipeline could not be found."
    )

    st.stop()


# ------------------------------------------------------------
# Backend import
# ------------------------------------------------------------

if OUTPUTS_PATH not in sys.path:

    sys.path.insert(
        0,
        OUTPUTS_PATH
    )


from army_provost_dss_backend import (
    execute_dashboard_dss,
    taxonomy_df
)


# ============================================================
# PERSISTENT DSS AUDIT LOGGING
# ============================================================

AUDIT_LOG_FIELDS = [
    "Decision ID",
    "Decision Timestamp",
    "Incident Type",
    "Location Description",
    "Domestic",
    "Year",
    "Month",
    "Day",
    "Hour",
    "District",
    "Beat",
    "Ward",
    "Community Area",
    "Provost Incident Category",
    "Incident Subcategory",
    "Priority",
    "Arrest Probability (%)",
    "ML Assessment",
    "Recommended Response Type",
    "Immediate Operator Guidance",
    "Coordination / Notification",
    "Scene / Evidence Considerations",
    "Escalation / Follow-up",
    "Guidance Status"
]


def append_dss_audit_record(audit_record):

    os.makedirs(
        LOGS_PATH,
        exist_ok=True
    )

    file_exists = os.path.exists(
        AUDIT_LOG_PATH
    )

    file_has_content = (
        file_exists
        and os.path.getsize(
            AUDIT_LOG_PATH
        ) > 0
    )

    with open(
        AUDIT_LOG_PATH,
        "a",
        newline="",
        encoding="utf-8"
    ) as audit_file:

        writer = csv.DictWriter(
            audit_file,
            fieldnames=AUDIT_LOG_FIELDS,
            extrasaction="ignore"
        )

        if not file_has_content:

            writer.writeheader()

        writer.writerow(
            audit_record
        )


# ============================================================
# CACHED DASHBOARD RESOURCES
# ============================================================

@st.cache_resource
def load_dashboard_preprocessor():

    return joblib.load(
        PREPROCESSOR_PATH
    )


def get_location_categories():

    preprocessor = (
        load_dashboard_preprocessor()
    )

    categorical_transformer = (
        preprocessor
        .named_transformers_[
            "categorical"
        ]
    )

    if hasattr(
        categorical_transformer,
        "named_steps"
    ):

        onehot_encoder = (
            categorical_transformer
            .named_steps[
                "onehot"
            ]
        )

    else:

        onehot_encoder = (
            categorical_transformer
        )

    categorical_columns = list(
        preprocessor
        .transformers_[0][2]
    )

    if (
        "Location Description"
        not in categorical_columns
    ):

        raise RuntimeError(
            "Location Description was not found "
            "in the fitted preprocessing pipeline."
        )

    location_index = (
        categorical_columns.index(
            "Location Description"
        )
    )

    location_categories = sorted(
        {
            str(value).strip()
            for value in
            onehot_encoder
            .categories_[
                location_index
            ]
            if value is not None
            and str(value).strip()
        }
    )

    return location_categories


# ============================================================
# PAGE CONFIGURATION
# ============================================================

st.set_page_config(
    page_title="Army Provost DSS",
    page_icon="🛡️",
    layout="wide",
    initial_sidebar_state="collapsed"
)


# ============================================================
# VISUAL FOUNDATION
# ============================================================

st.html(
    """
    <style>

    .stApp {
        background-color: #f4f6f8;
    }

    .block-container {
        max-width: 1500px;
        padding-top: 1.2rem;
        padding-bottom: 3rem;
    }

    #MainMenu {
        visibility: hidden;
    }

    footer {
        visibility: hidden;
    }

    header {
        background-color: transparent;
    }

    .command-header {
        background: linear-gradient(
            135deg,
            #111820 0%,
            #192630 55%,
            #223541 100%
        );

        border: 1px solid #344854;
        border-radius: 10px;

        padding: 25px 30px;
        margin-bottom: 15px;

        box-shadow:
            0px 3px 10px rgba(0,0,0,0.14);
    }

    .command-label {
        color: #9aabb5;
        font-size: 11px;
        font-weight: 700;
        letter-spacing: 0.16em;
        margin-bottom: 6px;
    }

    .command-title {
        color: #ffffff;
        font-size: 31px;
        font-weight: 700;
        margin-bottom: 5px;
    }

    .command-subtitle {
        color: #bdc8ce;
        font-size: 14px;
    }

    .status-strip {
        display: flex;
        gap: 10px;
        flex-wrap: wrap;
        margin-bottom: 23px;
    }

    .status-item {
        background-color: white;
        border: 1px solid #d7dde1;
        border-radius: 6px;

        padding: 8px 13px;

        font-size: 11px;
        font-weight: 700;

        color: #384750;

        box-shadow:
            0 1px 2px rgba(0,0,0,0.04);
    }

    .dot-green {
        display: inline-block;
        width: 8px;
        height: 8px;
        margin-right: 6px;

        border-radius: 50%;
        background-color: #268b57;
    }

    .dot-blue {
        display: inline-block;
        width: 8px;
        height: 8px;
        margin-right: 6px;

        border-radius: 50%;
        background-color: #467b9e;
    }

    .dot-amber {
        display: inline-block;
        width: 8px;
        height: 8px;
        margin-right: 6px;

        border-radius: 50%;
        background-color: #b2812d;
    }

    div[data-testid="stTextInput"] label,
    div[data-testid="stNumberInput"] label,
    div[data-testid="stSelectbox"] label {
        font-weight: 600;
        color: #35434d;
    }

    div.stButton > button {
        background-color: #263f4e;
        color: #ffffff;

        border: 1px solid #263f4e;
        border-radius: 6px;

        height: 48px;

        font-weight: 700;
        letter-spacing: 0.04em;
    }

    div.stButton > button:hover {
        background-color: #1d303b;
        color: white;
        border-color: #1d303b;
    }

    hr {
        border-color: #d8dfe3;
    }

    </style>
    """
)


# ============================================================
# COMMAND HEADER
# ============================================================

header_html = (
    '<div class="command-header">'

    '<div class="command-label">'
    'CONTROL ROOM DECISION SUPPORT'
    '</div>'

    '<div class="command-title">'
    'Army Provost Decision Support System'
    '</div>'

    '<div class="command-subtitle">'
    'ML-enabled incident classification, assessment and '
    'response-pathway recommendation'
    '</div>'

    '</div>'
)

st.html(
    header_html
)


# ------------------------------------------------------------
# System status strip
# ------------------------------------------------------------

status_html = (
    '<div class="status-strip">'

    '<div class="status-item">'
    '<span class="dot-green"></span>'
    'DSS BACKEND ONLINE'
    '</div>'

    '<div class="status-item">'
    '<span class="dot-blue"></span>'
    'RANDOM FOREST ACTIVE'
    '</div>'

    '<div class="status-item">'
    'DECISION THRESHOLD 0.50'
    '</div>'

    '<div class="status-item">'
    '<span class="dot-amber"></span>'
    'HUMAN DECISION AUTHORITY'
    '</div>'

    '</div>'
)

st.html(
    status_html
)


# ============================================================
# PREPARE AUTHORITATIVE INPUT OPTIONS
# ============================================================

try:

    incident_types = sorted(
        taxonomy_df[
            "Primary Type"
        ]
        .dropna()
        .astype(str)
        .str.strip()
        .unique()
        .tolist()
    )

    if not incident_types:

        raise RuntimeError(
            "Army Provost taxonomy contains "
            "no usable incident types."
        )


    location_categories = (
        get_location_categories()
    )

    if not location_categories:

        raise RuntimeError(
            "The fitted preprocessing pipeline contains "
            "no Location Description categories."
        )


except Exception as resource_error:

    st.error(
        "Unable to prepare dashboard input options: "
        f"{resource_error}"
    )

    st.stop()


# ============================================================
# 01 — INCIDENT INFORMATION
# ============================================================

st.subheader(
    "01  ·  Incident Information"
)

st.divider()


col1, col2, col3 = st.columns(
    [1.2, 1.2, 0.8]
)


with col1:

    default_incident_index = (
        incident_types.index(
            "BATTERY"
        )
        if "BATTERY" in incident_types
        else 0
    )

    primary_type = st.selectbox(
        "Incident Type",
        options=incident_types,
        index=default_incident_index,
        help=(
            "Select an incident type from the validated "
            "Army Provost incident taxonomy."
        )
    )


with col2:

    default_location_index = (
        location_categories.index(
            "STREET"
        )
        if "STREET" in location_categories
        else 0
    )

    location_description = st.selectbox(
        "Location Description",
        options=location_categories,
        index=default_location_index,
        help=(
            "Select a location category recognized by the "
            "fitted machine-learning preprocessing pipeline."
        )
    )


with col3:

    domestic = st.selectbox(
        "Domestic Incident",
        options=[
            False,
            True
        ],
        format_func=lambda x: (
            "Yes"
            if x
            else "No"
        )
    )


# ============================================================
# 02 — DATE & TIME
# ============================================================

st.subheader(
    "02  ·  Date & Time"
)

st.divider()


col1, col2, col3, col4 = (
    st.columns(4)
)


with col1:

    year = st.number_input(
        "Year",
        min_value=2001,
        max_value=2100,
        value=2025,
        step=1
    )


with col2:

    month = st.number_input(
        "Month",
        min_value=1,
        max_value=12,
        value=7,
        step=1
    )


with col3:

    day = st.number_input(
        "Day",
        min_value=1,
        max_value=31,
        value=15,
        step=1
    )


with col4:

    hour = st.number_input(
        "Hour",
        min_value=0,
        max_value=23,
        value=20,
        step=1
    )


# ============================================================
# 03 — ADMINISTRATIVE INFORMATION
# ============================================================

st.subheader(
    "03  ·  Administrative Information"
)

st.divider()


col1, col2, col3, col4 = (
    st.columns(4)
)


with col1:

    district = st.number_input(
        "District",
        min_value=0,
        value=1,
        step=1
    )


with col2:

    beat = st.number_input(
        "Beat",
        min_value=0,
        value=100,
        step=1
    )


with col3:

    ward = st.number_input(
        "Ward",
        min_value=0,
        value=1,
        step=1
    )


with col4:

    community_area = st.number_input(
        "Community Area",
        min_value=0,
        value=32,
        step=1
    )


# ============================================================
# ANALYSIS CONTROL
# ============================================================

st.write("")


analyze_button = st.button(
    "ANALYZE INCIDENT",
    type="primary",
    use_container_width=True
)


# ============================================================
# DSS EXECUTION
# ============================================================

if analyze_button:

    try:

        result = execute_dashboard_dss(
            primary_type=primary_type,
            location_description=location_description,
            domestic=domestic,
            year=year,
            month=month,
            day=day,
            hour=hour,
            district=district,
            beat=beat,
            ward=ward,
            community_area=community_area
        )


        # ----------------------------------------------------
        # Result extraction
        # ----------------------------------------------------

        validated_input = result[
            "Validated Input"
        ]

        dss_result = result[
            "DSS Result"
        ]

        decision = dss_result[
            "Decision"
        ]

        record = dss_result[
            "Decision Record"
        ]

        interpretation = dss_result[
            "Interpretation"
        ]

        operator_summary = dss_result[
            "Operator Summary"
        ]

        response_guidance = dss_result[
            "Response Guidance"
        ]


        # ----------------------------------------------------
        # Store session-level Recent Decision record
        # ----------------------------------------------------

        recent_decision = {
            "Decision ID": record[
                "Decision ID"
            ],
            "Timestamp": record[
                "Decision Timestamp"
            ],
            "Incident Type": decision[
                "Primary Type"
            ],
            "Priority": decision[
                "Priority"
            ],
            "Arrest Probability (%)": decision[
                "Arrest Probability (%)"
            ],
            "ML Assessment": decision[
                "Arrest Prediction"
            ],
            "Recommended Response Type": decision[
                "Recommended Response Type"
            ]
        }

        st.session_state.recent_decisions.insert(
            0,
            recent_decision
        )

        st.session_state.recent_decisions = (
            st.session_state.recent_decisions[
                :MAX_RECENT_DECISIONS
            ]
        )


        # ----------------------------------------------------
        # Persist DSS audit record
        # ----------------------------------------------------

        audit_record = {
            "Decision ID": record[
                "Decision ID"
            ],
            "Decision Timestamp": record[
                "Decision Timestamp"
            ],
            "Incident Type": decision[
                "Primary Type"
            ],
            "Location Description": location_description,
            "Domestic": domestic,
            "Year": year,
            "Month": month,
            "Day": day,
            "Hour": hour,
            "District": district,
            "Beat": beat,
            "Ward": ward,
            "Community Area": community_area,
            "Provost Incident Category": decision[
                "Provost Incident Category"
            ],
            "Incident Subcategory": decision[
                "Incident Subcategory"
            ],
            "Priority": decision[
                "Priority"
            ],
            "Arrest Probability (%)": decision[
                "Arrest Probability (%)"
            ],
            "ML Assessment": decision[
                "Arrest Prediction"
            ],
            "Recommended Response Type": decision[
                "Recommended Response Type"
            ],
            "Immediate Operator Guidance": response_guidance[
                "Immediate Operator Guidance"
            ],
            "Coordination / Notification": response_guidance[
                "Coordination / Notification"
            ],
            "Scene / Evidence Considerations": response_guidance[
                "Scene / Evidence Considerations"
            ],
            "Escalation / Follow-up": response_guidance[
                "Escalation / Follow-up"
            ],
            "Guidance Status": response_guidance[
                "Guidance Status"
            ]
        }


        audit_log_written = True
        audit_log_error = None

        try:

            append_dss_audit_record(
                audit_record
            )

        except Exception as audit_exception:

            audit_log_written = False
            audit_log_error = str(
                audit_exception
            )


        # ----------------------------------------------------
        # Successful execution indicator
        # ----------------------------------------------------

        st.success(
            "Incident successfully analyzed."
        )

        if audit_log_written:

            st.caption(
                "Persistent DSS audit record saved successfully."
            )

        else:

            st.warning(
                "The incident analysis succeeded, but the "
                "persistent DSS audit record could not be saved. "
                f"Audit logging error: {audit_log_error}"
            )


        # ====================================================
        # 04 — PROVOST ASSESSMENT
        # ====================================================

        st.subheader(
            "04  ·  Provost Assessment"
        )

        st.divider()


        category = decision[
            "Provost Incident Category"
        ]

        subcategory = decision[
            "Incident Subcategory"
        ]

        priority = decision[
            "Priority"
        ]


        col1, col2, col3 = (
            st.columns(3)
        )


        with col1:

            with st.container(
                border=True
            ):

                st.caption(
                    "PROVOST CATEGORY"
                )

                st.markdown(
                    f"### {category}"
                )


        with col2:

            with st.container(
                border=True
            ):

                st.caption(
                    "INCIDENT SUBCATEGORY"
                )

                st.markdown(
                    f"### {subcategory}"
                )


        with col3:

            with st.container(
                border=True
            ):

                st.caption(
                    "PRIORITY"
                )

                if priority == "Critical":

                    st.error(
                        "CRITICAL"
                    )

                elif priority == "High":

                    st.warning(
                        "HIGH"
                    )

                elif priority == "Moderate":

                    st.info(
                        "MODERATE"
                    )

                else:

                    st.success(
                        "LOW"
                    )


        # ====================================================
        # 05 — MACHINE LEARNING ASSESSMENT
        # ====================================================

        st.subheader(
            "05  ·  Machine Learning Assessment"
        )

        st.divider()


        probability = decision[
            "Arrest Probability (%)"
        ]

        prediction = decision[
            "Arrest Prediction"
        ]


        col1, col2, col3 = (
            st.columns(3)
        )


        with col1:

            with st.container(
                border=True
            ):

                st.caption(
                    "ARREST PROBABILITY"
                )

                st.markdown(
                    f"# {probability:.2f}%"
                )


        with col2:

            with st.container(
                border=True
            ):

                st.caption(
                    "ML ASSESSMENT"
                )

                st.markdown(
                    f"### {prediction}"
                )


        with col3:

            with st.container(
                border=True
            ):

                st.caption(
                    "DECISION THRESHOLD"
                )

                st.markdown(
                    "### 50.00%"
                )


        st.info(
            """
            **How to interpret this assessment**

            Arrest Probability estimates the historical likelihood
            of an arrest being recorded for incidents with similar
            characteristics.

            It does **not** represent incident severity or operational
            importance. A serious or Critical/High-priority incident
            may still receive a **Less Likely Arrest** ML assessment.

            **Official incident priority always remains separate from
            the ML arrest-probability assessment.**
            """
        )


        # ====================================================
        # 06 — RECOMMENDED RESPONSE PATHWAY
        # ====================================================

        st.subheader(
            "06  ·  Recommended Response Pathway"
        )

        st.divider()


        response_type = response_guidance[
            "Recommended Response Type"
        ]

        immediate_guidance = response_guidance[
            "Immediate Operator Guidance"
        ]

        coordination_guidance = response_guidance[
            "Coordination / Notification"
        ]

        scene_guidance = response_guidance[
            "Scene / Evidence Considerations"
        ]

        escalation_guidance = response_guidance[
            "Escalation / Follow-up"
        ]

        guidance_status = response_guidance[
            "Guidance Status"
        ]


        # ----------------------------------------------------
        # Broad response pathway
        # ----------------------------------------------------

        with st.container(
            border=True
        ):

            st.caption(
                "RECOMMENDED RESPONSE TYPE"
            )

            st.markdown(
                f"### {response_type}"
            )


        # ----------------------------------------------------
        # Immediate operator guidance
        # ----------------------------------------------------

        with st.container(
            border=True
        ):

            st.caption(
                "IMMEDIATE OPERATOR GUIDANCE"
            )

            st.info(
                immediate_guidance
            )


        # ----------------------------------------------------
        # Coordination + Scene considerations
        # ----------------------------------------------------

        col1, col2 = st.columns(2)


        with col1:

            with st.container(
                border=True
            ):

                st.caption(
                    "COORDINATION / NOTIFICATION"
                )

                st.write(
                    coordination_guidance
                )


        with col2:

            with st.container(
                border=True
            ):

                st.caption(
                    "SCENE / EVIDENCE CONSIDERATIONS"
                )

                st.write(
                    scene_guidance
                )


        # ----------------------------------------------------
        # Escalation / follow-up
        # ----------------------------------------------------

        with st.container(
            border=True
        ):

            st.caption(
                "ESCALATION / FOLLOW-UP"
            )

            st.write(
                escalation_guidance
            )


        st.caption(
            f"{guidance_status}. "
            "This guidance supports authorized operator judgment "
            "and is not represented as official operational SOP."
        )


        # ====================================================
        # 07 — DECISION RECORD
        # ====================================================

        st.subheader(
            "07  ·  Decision Record"
        )

        st.divider()


        col1, col2 = (
            st.columns(2)
        )


        with col1:

            with st.container(
                border=True
            ):

                st.caption(
                    "DECISION ID"
                )

                st.code(
                    record[
                        "Decision ID"
                    ],
                    language=None
                )


        with col2:

            with st.container(
                border=True
            ):

                st.caption(
                    "DECISION TIMESTAMP"
                )

                st.code(
                    record[
                        "Decision Timestamp"
                    ],
                    language=None
                )


        # ====================================================
        # DECISION INTERPRETATION
        # ====================================================

        with st.expander(
            "Decision Interpretation"
        ):

            for key, value in (
                interpretation.items()
            ):

                st.markdown(
                    f"**{key}**"
                )

                st.write(
                    value
                )

                st.write("")


        # ====================================================
        # OPERATOR SUMMARY
        # ====================================================

        with st.expander(
            "Operator Summary"
        ):

            st.text(
                operator_summary
            )


        # ====================================================
        # HUMAN OVERSIGHT
        # ====================================================

        st.warning(
            """
            **Human Oversight Requirement**

            This system provides decision-support recommendations
            for authorized operator consideration.

            Official incident priority and machine-learning arrest
            probability are separate decision-support indicators.

            The ML result does not override the established incident
            priority.

            Structured response guidance is prototype decision-support
            guidance and is not represented as official operational SOP.

            Final operational decisions remain with the authorized
            operator.
            """
        )


    except Exception as e:

        st.error(
            "Unable to analyze the incident: "
            f"{e}"
        )


# ============================================================
# 08 — RECENT DECISIONS
# ============================================================

st.subheader(
    "08  ·  Recent Decisions"
)

st.divider()


if st.session_state.recent_decisions:

    st.caption(
        "Session-level decision history. "
        "Latest decision appears first. "
        "These table rows are cleared when the Streamlit session ends. "
        "Persistent DSS audit records are stored separately in the "
        "project Logs directory."
    )

    st.dataframe(
        st.session_state.recent_decisions,
        use_container_width=True,
        hide_index=True
    )

else:

    st.info(
        "No decisions have been generated in this session yet."
    )
'''


# ============================================================
# WRITE DASHBOARD APPLICATION
# ============================================================

with open(
    DASHBOARD_APP_PATH,
    "w",
    encoding="utf-8"
) as f:

    f.write(
        dashboard_code
    )


# ============================================================
# VERIFY GENERATED APPLICATION
# ============================================================

file_exists = os.path.exists(
    DASHBOARD_APP_PATH
)


file_size = (
    os.path.getsize(
        DASHBOARD_APP_PATH
    )
    if file_exists
    else 0
)


print("=" * 80)

print(
    "ARMY PROVOST DASHBOARD — "
    "SESSION DECISION HISTORY UPDATE"
)

print("=" * 80)


print(
    f"Dashboard application : "
    f"{DASHBOARD_APP_PATH}"
)

print(
    f"File exists           : "
    f"{file_exists}"
)

print(
    f"File size             : "
    f"{file_size / 1024:.2f} KB"
)


if not file_exists:

    raise RuntimeError(
        "Dashboard application file "
        "was not created."
    )


print("\n" + "=" * 80)

print(
    "✅ DASHBOARD APPLICATION UPDATED SUCCESSFULLY"
)

print("=" * 80)

print(
    "\nIncident Type source       : "
    "Validated Army Provost taxonomy"
)

print(
    "Location Description source : "
    "Fitted preprocessing pipeline"
)

print(
    "Response Guidance source    : "
    "Persistent DSS backend"
)

print(
    "HTML/CSS rendering          : "
    "st.html()"
)

print(
    "Default Incident Type       : "
    "BATTERY"
)

print(
    "Default Location            : "
    "STREET"
)

print(
    "Recent Decision History     : "
    "st.session_state (max 20 records)"
)

print(
    "Persistent DSS Audit Log    : "
    "/content/drive/MyDrive/Army_Provost_ML_Project/"
    "Logs/Army_Provost_DSS_Audit_Log.csv"
)

print(
    "Audit Log Write Behavior    : "
    "Append one row per successful analysis"
)

print("=" * 80)

ARMY PROVOST DASHBOARD — SESSION DECISION HISTORY UPDATE
Dashboard application : /content/drive/MyDrive/Army_Provost_ML_Project/Outputs/army_provost_dashboard.py
File exists           : True
File size             : 28.23 KB

✅ DASHBOARD APPLICATION UPDATED SUCCESSFULLY

Incident Type source       : Validated Army Provost taxonomy
Location Description source : Fitted preprocessing pipeline
Response Guidance source    : Persistent DSS backend
HTML/CSS rendering          : st.html()
Default Incident Type       : BATTERY
Default Location            : STREET
Recent Decision History     : st.session_state (max 20 records)
Persistent DSS Audit Log    : /content/drive/MyDrive/Army_Provost_ML_Project/Logs/Army_Provost_DSS_Audit_Log.csv
Audit Log Write Behavior    : Append one row per successful analysis


In [6]:
# ============================================================
# ARMY PROVOST DASHBOARD
# SECTION 4 — APPLICATION VALIDATION
# ============================================================

import ast
import os

print("=" * 80)
print("ARMY PROVOST DASHBOARD — APPLICATION VALIDATION")
print("=" * 80)

# ------------------------------------------------------------
# Verify application file
# ------------------------------------------------------------

if not os.path.exists(DASHBOARD_APP_PATH):

    raise FileNotFoundError(
        f"Dashboard application not found:\n{DASHBOARD_APP_PATH}"
    )

print(f"Application file : {DASHBOARD_APP_PATH}")
print("File exists      : True")

# ------------------------------------------------------------
# Read application source
# ------------------------------------------------------------

with open(
    DASHBOARD_APP_PATH,
    "r",
    encoding="utf-8"
) as f:

    dashboard_source = f.read()

print(
    f"Source size      : "
    f"{len(dashboard_source):,} characters"
)

# ------------------------------------------------------------
# Python syntax validation
# ------------------------------------------------------------

try:

    ast.parse(
        dashboard_source,
        filename=DASHBOARD_APP_PATH
    )

    syntax_valid = True

except SyntaxError as e:

    syntax_valid = False

    print("\n❌ SYNTAX ERROR")
    print(e)

# ------------------------------------------------------------
# Required dashboard components
# ------------------------------------------------------------

required_components = [
    "streamlit",
    "execute_dashboard_dss",
    "ANALYZE INCIDENT",
    "Provost Incident Category",
    "Incident Subcategory",
    "Arrest Probability",
    "Arrest Prediction",
    "Recommended Response Type",
    "Decision ID",
    "Decision Timestamp"
]

component_results = {}

for component in required_components:

    component_results[component] = (
        component in dashboard_source
    )

# ------------------------------------------------------------
# Validation summary
# ------------------------------------------------------------

print("\n" + "=" * 80)
print("VALIDATION SUMMARY")
print("=" * 80)

print(
    f"Python syntax valid : "
    f"{syntax_valid}"
)

print("\nRequired components:")

for component, present in component_results.items():

    status = "✅" if present else "❌"

    print(
        f"{status} {component}"
    )

all_components_present = all(
    component_results.values()
)

print("\n" + "=" * 80)

if syntax_valid and all_components_present:

    print("✅ DASHBOARD APPLICATION VALIDATION PASSED")

else:

    print("❌ DASHBOARD APPLICATION VALIDATION FAILED")

print("=" * 80)

ARMY PROVOST DASHBOARD — APPLICATION VALIDATION
Application file : /content/drive/MyDrive/Army_Provost_ML_Project/Outputs/army_provost_dashboard.py
File exists      : True
Source size      : 28,881 characters

VALIDATION SUMMARY
Python syntax valid : True

Required components:
✅ streamlit
✅ execute_dashboard_dss
✅ ANALYZE INCIDENT
✅ Provost Incident Category
✅ Incident Subcategory
✅ Arrest Probability
✅ Arrest Prediction
✅ Recommended Response Type
✅ Decision ID
✅ Decision Timestamp

✅ DASHBOARD APPLICATION VALIDATION PASSED


In [8]:
# ============================================================
# ARMY PROVOST DASHBOARD
# SECTION 5A — START / REUSE STREAMLIT SERVER
# ============================================================

import subprocess
import time
import socket

STREAMLIT_PORT = 8501

print("=" * 80)
print("ARMY PROVOST DASHBOARD — STREAMLIT SERVER")
print("=" * 80)


# ------------------------------------------------------------
# Helper: check whether Streamlit port is already active
# ------------------------------------------------------------

def is_port_open(host="127.0.0.1", port=8501):
    with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as sock:
        sock.settimeout(0.5)
        return sock.connect_ex((host, port)) == 0


# ------------------------------------------------------------
# Reuse existing server if already running
# ------------------------------------------------------------

if is_port_open(port=STREAMLIT_PORT):

    print("✅ Existing Streamlit server detected.")
    print(f"Port   : {STREAMLIT_PORT}")
    print("Status : RUNNING")
    print("\nNo new Streamlit process was started.")

else:

    print("No active Streamlit server detected.")
    print("Starting Streamlit...")

    streamlit_process = subprocess.Popen(
        [
            "streamlit",
            "run",
            DASHBOARD_APP_PATH,
            "--server.port",
            str(STREAMLIT_PORT),
            "--server.address",
            "0.0.0.0",
            "--server.headless",
            "true"
        ],
        stdout=subprocess.DEVNULL,
        stderr=subprocess.STDOUT
    )

    # --------------------------------------------------------
    # Wait only until the server is actually ready
    # --------------------------------------------------------

    server_ready = False

    for _ in range(20):

        if is_port_open(port=STREAMLIT_PORT):
            server_ready = True
            break

        # Detect an unexpected process failure
        if streamlit_process.poll() is not None:
            break

        time.sleep(0.5)

    # --------------------------------------------------------
    # Result
    # --------------------------------------------------------

    if server_ready:

        print("✅ Streamlit server started successfully.")
        print(f"Port   : {STREAMLIT_PORT}")
        print("Status : RUNNING")

    else:

        raise RuntimeError(
            "Streamlit server did not become available on port 8501."
        )

print("=" * 80)

ARMY PROVOST DASHBOARD — STREAMLIT SERVER
No active Streamlit server detected.
Starting Streamlit...
✅ Streamlit server started successfully.
Port   : 8501
Status : RUNNING


In [12]:
# ============================================================
# ARMY PROVOST DASHBOARD
# INSTALL CLOUDFLARED — ONE TIME PER COLAB RUNTIME
# ============================================================

import os
import subprocess

print("=" * 80)
print("CLOUDFLARE TUNNEL — DEPENDENCY CHECK")
print("=" * 80)

CLOUDFLARED_PATH = "/usr/local/bin/cloudflared"

# ------------------------------------------------------------
# Check whether cloudflared already exists
# ------------------------------------------------------------

if os.path.exists(CLOUDFLARED_PATH):

    print("✅ cloudflared is already installed.")

else:

    print("cloudflared not found.")
    print("Installing cloudflared...")

    subprocess.run(
        [
            "wget",
            "-q",
            "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64",
            "-O",
            CLOUDFLARED_PATH
        ],
        check=True
    )

    subprocess.run(
        [
            "chmod",
            "+x",
            CLOUDFLARED_PATH
        ],
        check=True
    )

    print("✅ cloudflared installed successfully.")


# ------------------------------------------------------------
# Verify installation
# ------------------------------------------------------------

version_result = subprocess.run(
    [
        CLOUDFLARED_PATH,
        "--version"
    ],
    capture_output=True,
    text=True,
    check=True
)

print(
    "\nVersion:",
    version_result.stdout.strip()
)

print("\n" + "=" * 80)
print("✅ CLOUDFLARE TUNNEL DEPENDENCY READY")
print("=" * 80)

CLOUDFLARE TUNNEL — DEPENDENCY CHECK
✅ cloudflared is already installed.

Version: cloudflared version 2026.7.3 (built 2026-07-23-09:58 UTC)

✅ CLOUDFLARE TUNNEL DEPENDENCY READY


In [9]:
# ============================================================
# ARMY PROVOST DASHBOARD
# SECTION 5B — CREATE / REUSE CLOUDFLARE QUICK TUNNEL
# ============================================================

import subprocess
import time
import re
import os

print("=" * 80)
print("ARMY PROVOST DASHBOARD — CLOUDFLARE PUBLIC ACCESS")
print("=" * 80)

CLOUDFLARED_PATH = "/usr/local/bin/cloudflared"
STREAMLIT_URL = "http://127.0.0.1:8501"


# ------------------------------------------------------------
# Check whether an existing Cloudflare tunnel process exists
# ------------------------------------------------------------

existing_tunnel = subprocess.run(
    ["pgrep", "-af", "cloudflared tunnel --url"],
    capture_output=True,
    text=True
)

tunnel_running = bool(
    existing_tunnel.stdout.strip()
)


# ------------------------------------------------------------
# Reuse existing tunnel if URL is still known
# ------------------------------------------------------------

if (
    tunnel_running
    and "dashboard_url" in globals()
    and dashboard_url
    and "trycloudflare.com" in dashboard_url
):

    print("✅ Existing Cloudflare tunnel detected.")
    print("Status : RUNNING")

    print(
        "\nReusing existing dashboard URL:\n"
    )

    print(
        dashboard_url
    )


else:

    # --------------------------------------------------------
    # Clean stale Cloudflare tunnel if necessary
    # --------------------------------------------------------

    if tunnel_running:

        print(
            "Existing Cloudflare tunnel process detected, "
            "but its URL is unavailable."
        )

        print(
            "Restarting tunnel cleanly..."
        )

        subprocess.run(
            [
                "pkill",
                "-f",
                "cloudflared tunnel --url"
            ],
            stdout=subprocess.DEVNULL,
            stderr=subprocess.DEVNULL
        )

        time.sleep(1)

    else:

        print(
            "No active Cloudflare tunnel detected."
        )

        print(
            "Starting Cloudflare Quick Tunnel..."
        )


    # --------------------------------------------------------
    # Start Cloudflare Quick Tunnel
    # --------------------------------------------------------

    cloudflare_process = subprocess.Popen(
        [
            CLOUDFLARED_PATH,
            "tunnel",
            "--url",
            STREAMLIT_URL,
            "--no-autoupdate"
        ],
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1
    )


    # --------------------------------------------------------
    # Capture generated trycloudflare.com URL
    # --------------------------------------------------------

    dashboard_url = None

    start_time = time.time()

    while (
        time.time() - start_time
        < 30
    ):

        line = (
            cloudflare_process
            .stdout
            .readline()
        )

        if line:

            clean_line = (
                line.strip()
            )

            if clean_line:

                print(
                    clean_line
                )

            match = re.search(
                r"https://[a-zA-Z0-9\-]+\.trycloudflare\.com",
                clean_line
            )

            if match:

                dashboard_url = (
                    match.group(0)
                )

                break


        if (
            cloudflare_process.poll()
            is not None
        ):

            break


        time.sleep(
            0.25
        )


    # --------------------------------------------------------
    # Validate tunnel URL
    # --------------------------------------------------------

    if not dashboard_url:

        raise RuntimeError(
            "Cloudflare tunnel started but no "
            "trycloudflare.com URL could be obtained."
        )


    print(
        "\n✅ New Cloudflare Quick Tunnel established."
    )


# ------------------------------------------------------------
# Final status
# ------------------------------------------------------------

print(
    "\n" + "=" * 80
)

print(
    "DASHBOARD ACCESS READY"
)

print(
    "=" * 80
)


print(
    "\nOpen this URL in your browser:\n"
)

print(
    dashboard_url
)


print(
    "\nStreamlit Port : 8501"
)

print(
    "Tunnel Type    : Cloudflare Quick Tunnel"
)

print(
    "Tunnel Status  : RUNNING"
)

print(
    "=" * 80
)

ARMY PROVOST DASHBOARD — CLOUDFLARE PUBLIC ACCESS
No active Cloudflare tunnel detected.
Starting Cloudflare Quick Tunnel...
2026-08-09T17:19:49Z INF Thank you for trying Cloudflare Tunnel. Doing so, without a Cloudflare account, is a quick way to experiment and try it out. However, be aware that these account-less Tunnels have no uptime guarantee, are subject to the Cloudflare Online Services Terms of Use (https://www.cloudflare.com/website-terms/), and Cloudflare reserves the right to investigate your use of Tunnels for violations of such terms. If you intend to use Tunnels in production you should use a pre-created named tunnel by following: https://developers.cloudflare.com/cloudflare-one/connections/connect-apps
2026-08-09T17:19:49Z INF Requesting new quick Tunnel on trycloudflare.com...
2026-08-09T17:19:58Z INF +--------------------------------------------------------------------------------------------+
2026-08-09T17:19:58Z INF |  Your quick Tunnel has been created! Visit it at (

In [16]:
# ============================================================
# PHASE A — FINAL FUNCTIONAL VALIDATION
# Six Representative Incident Scenarios
# CORRECTED FOR CURRENT DSS RESULT STRUCTURE
# ============================================================

import sys
import importlib
import pandas as pd

PROJECT_ROOT = "/content/drive/MyDrive/Army_Provost_ML_Project"
OUTPUTS_PATH = f"{PROJECT_ROOT}/Outputs"

if OUTPUTS_PATH not in sys.path:
    sys.path.insert(0, OUTPUTS_PATH)

import army_provost_dss_backend
importlib.reload(army_provost_dss_backend)

from army_provost_dss_backend import execute_dashboard_dss


# ------------------------------------------------------------
# Representative Version 1.0 validation scenarios
# ------------------------------------------------------------

validation_incidents = [
    "BATTERY",
    "ARSON",
    "THEFT",
    "NARCOTICS",
    "WEAPONS VIOLATION",
    "HOMICIDE"
]


# ------------------------------------------------------------
# Established dashboard/default regression inputs
# ------------------------------------------------------------

COMMON_INPUTS = {
    "location_description": "STREET",
    "domestic": False,
    "year": 2025,
    "month": 7,
    "day": 15,
    "hour": 20,
    "district": 1,
    "beat": 100,
    "ward": 1,
    "community_area": 32
}


# ------------------------------------------------------------
# Execute scenarios
# ------------------------------------------------------------

validation_rows = []

print("=" * 78)
print("ARMY PROVOST DSS — PHASE A FINAL FUNCTIONAL VALIDATION")
print("=" * 78)


for incident_type in validation_incidents:

    try:

        result = execute_dashboard_dss(
            primary_type=incident_type,
            **COMMON_INPUTS
        )

        # ----------------------------------------------------
        # Correct current backend structure
        # ----------------------------------------------------

        dss_result = result["DSS Result"]

        decision = dss_result[
            "Decision"
        ]

        record = dss_result[
            "Decision Record"
        ]

        guidance = dss_result[
            "Response Guidance"
        ]


        # ----------------------------------------------------
        # Guidance completeness
        # ----------------------------------------------------

        required_guidance_fields = [
            "Immediate Operator Guidance",
            "Coordination / Notification",
            "Scene / Evidence Considerations",
            "Escalation / Follow-up"
        ]

        guidance_complete = all(
            bool(
                str(
                    guidance.get(
                        field,
                        ""
                    )
                ).strip()
            )
            for field in required_guidance_fields
        )


        # ----------------------------------------------------
        # Decision record completeness
        # ----------------------------------------------------

        decision_record_complete = (
            bool(
                str(
                    record.get(
                        "Decision ID",
                        ""
                    )
                ).strip()
            )
            and
            bool(
                str(
                    record.get(
                        "Decision Timestamp",
                        ""
                    )
                ).strip()
            )
        )


        # ----------------------------------------------------
        # Store successful validation result
        # ----------------------------------------------------

        validation_rows.append({

            "Incident Type":
                incident_type,

            "Provost Category":
                decision[
                    "Provost Incident Category"
                ],

            "Subcategory":
                decision[
                    "Incident Subcategory"
                ],

            "Priority":
                decision[
                    "Priority"
                ],

            "Arrest Probability (%)":
                round(
                    float(
                        decision[
                            "Arrest Probability (%)"
                        ]
                    ),
                    2
                ),

            "ML Assessment":
                decision[
                    "Arrest Prediction"
                ],

            "Recommended Response":
                decision[
                    "Recommended Response Type"
                ],

            "Guidance Complete":
                guidance_complete,

            "Decision Record Complete":
                decision_record_complete,

            "Execution Status":
                "PASS"
        })


    except Exception as exc:

        validation_rows.append({

            "Incident Type":
                incident_type,

            "Provost Category":
                "ERROR",

            "Subcategory":
                "ERROR",

            "Priority":
                "ERROR",

            "Arrest Probability (%)":
                None,

            "ML Assessment":
                "ERROR",

            "Recommended Response":
                "ERROR",

            "Guidance Complete":
                False,

            "Decision Record Complete":
                False,

            "Execution Status":
                f"FAIL — {exc}"
        })


# ------------------------------------------------------------
# Build validation table
# ------------------------------------------------------------

validation_df = pd.DataFrame(
    validation_rows
)

display(
    validation_df
)


# ------------------------------------------------------------
# Automated validation summary
# ------------------------------------------------------------

successful_scenarios = (
    validation_df[
        "Execution Status"
    ]
    == "PASS"
).sum()

guidance_passes = (
    validation_df[
        "Guidance Complete"
    ]
    == True
).sum()

record_passes = (
    validation_df[
        "Decision Record Complete"
    ]
    == True
).sum()


# ------------------------------------------------------------
# BATTERY golden regression check
# ------------------------------------------------------------

battery_rows = validation_df[
    validation_df[
        "Incident Type"
    ]
    == "BATTERY"
]

golden_regression_pass = False


if (
    not battery_rows.empty
    and
    battery_rows.iloc[0][
        "Execution Status"
    ]
    == "PASS"
):

    battery_row = (
        battery_rows.iloc[0]
    )

    battery_probability_pass = (
        abs(
            float(
                battery_row[
                    "Arrest Probability (%)"
                ]
            )
            - 39.57
        )
        <= 0.01
    )

    battery_prediction_pass = (
        battery_row[
            "ML Assessment"
        ]
        == "Less Likely Arrest"
    )

    golden_regression_pass = (
        battery_probability_pass
        and
        battery_prediction_pass
    )


# ------------------------------------------------------------
# Final summary
# ------------------------------------------------------------

print()

print("=" * 78)
print("VALIDATION SUMMARY")
print("=" * 78)

print(
    f"Scenario Execution        : "
    f"{successful_scenarios}/6 PASS"
)

print(
    f"Structured Guidance       : "
    f"{guidance_passes}/6 PASS"
)

print(
    f"Decision Record           : "
    f"{record_passes}/6 PASS"
)

print(
    "BATTERY Golden Regression : "
    + (
        "PASS"
        if golden_regression_pass
        else "FAIL"
    )
)

print("-" * 78)


if (
    successful_scenarios == 6
    and
    guidance_passes == 6
    and
    record_passes == 6
    and
    golden_regression_pass
):

    print(
        "FINAL RESULT : PASS — "
        "All Phase A functional validation checks succeeded."
    )

else:

    print(
        "FINAL RESULT : REVIEW REQUIRED — "
        "One or more validation checks failed."
    )


print("=" * 78)

ARMY PROVOST DSS — PHASE A FINAL FUNCTIONAL VALIDATION


,Incident Type,Provost Category,Subcategory,Priority,Arrest Probability (%),ML Assessment,Recommended Response,Guidance Complete,Decision Record Complete,Execution Status
0,BATTERY,Personnel & Physical Safety,Physical Assault,High,39.57,Less Likely Arrest,Personnel Safety / Provost Response,True,True,PASS
1,ARSON,"Fire, Destructive & Major Threats",Arson / Fire Threat,High,40.57,Less Likely Arrest,Fire / Major Threat Response,True,True,PASS
2,THEFT,Property & Asset Security,Theft / Property Loss,Moderate,19.97,Less Likely Arrest,Property / Asset Security Response,True,True,PASS
3,NARCOTICS,Controlled Substances,Narcotics Activity,High,91.13,Likely Arrest,Controlled Substances / Security Response,True,True,PASS
4,WEAPONS VIOLATION,Weapons & Armed Security,Weapons Violation,High,79.82,Likely Arrest,Weapons / Security Response,True,True,PASS
5,HOMICIDE,Personnel & Physical Safety,Fatal Violence,Critical,51.77,Likely Arrest,Personnel Safety / Provost Response,True,True,PASS



VALIDATION SUMMARY
Scenario Execution        : 6/6 PASS
Structured Guidance       : 6/6 PASS
Decision Record           : 6/6 PASS
BATTERY Golden Regression : PASS
------------------------------------------------------------------------------
FINAL RESULT : PASS — All Phase A functional validation checks succeeded.


In [17]:
# ============================================================
# PHASE B — BOUNDARY & ERROR TESTING
# Army Provost DSS Version 1.0
# ============================================================

import sys
import importlib
import pandas as pd

PROJECT_ROOT = "/content/drive/MyDrive/Army_Provost_ML_Project"
OUTPUTS_PATH = f"{PROJECT_ROOT}/Outputs"

if OUTPUTS_PATH not in sys.path:
    sys.path.insert(0, OUTPUTS_PATH)

import army_provost_dss_backend
importlib.reload(army_provost_dss_backend)

from army_provost_dss_backend import execute_dashboard_dss


# ------------------------------------------------------------
# Normal baseline input
# ------------------------------------------------------------

BASE_INPUT = {
    "primary_type": "BATTERY",
    "location_description": "STREET",
    "domestic": False,
    "year": 2025,
    "month": 7,
    "day": 15,
    "hour": 20,
    "district": 1,
    "beat": 100,
    "ward": 1,
    "community_area": 32
}


# ------------------------------------------------------------
# Boundary / invalid test cases
#
# Expected:
# PASS_EXECUTION = valid input should execute successfully
# SAFE_REJECTION = invalid input should raise an exception safely
# ------------------------------------------------------------

test_cases = [
    {
        "Test": "Month lower boundary",
        "Expected": "PASS_EXECUTION",
        "Changes": {
            "month": 1
        }
    },
    {
        "Test": "Month upper boundary",
        "Expected": "PASS_EXECUTION",
        "Changes": {
            "month": 12
        }
    },
    {
        "Test": "Hour lower boundary",
        "Expected": "PASS_EXECUTION",
        "Changes": {
            "hour": 0
        }
    },
    {
        "Test": "Hour upper boundary",
        "Expected": "PASS_EXECUTION",
        "Changes": {
            "hour": 23
        }
    },
    {
        "Test": "Administrative zero boundary",
        "Expected": "PASS_EXECUTION",
        "Changes": {
            "district": 0,
            "beat": 0,
            "ward": 0,
            "community_area": 0
        }
    },
    {
        "Test": "Domestic = True",
        "Expected": "PASS_EXECUTION",
        "Changes": {
            "domestic": True
        }
    },
    {
        "Test": "Invalid month = 13",
        "Expected": "SAFE_REJECTION",
        "Changes": {
            "month": 13
        }
    },
    {
        "Test": "Invalid month = 0",
        "Expected": "SAFE_REJECTION",
        "Changes": {
            "month": 0
        }
    },
    {
        "Test": "Invalid hour = 24",
        "Expected": "SAFE_REJECTION",
        "Changes": {
            "hour": 24
        }
    },
    {
        "Test": "Unknown incident type",
        "Expected": "SAFE_REJECTION",
        "Changes": {
            "primary_type": "INVALID INCIDENT TYPE"
        }
    }
]


# ------------------------------------------------------------
# Execute tests
# ------------------------------------------------------------

results = []

print("=" * 80)
print("ARMY PROVOST DSS — PHASE B BOUNDARY & ERROR TESTING")
print("=" * 80)


for case in test_cases:

    test_input = BASE_INPUT.copy()
    test_input.update(
        case["Changes"]
    )

    expected = case["Expected"]

    try:

        result = execute_dashboard_dss(
            **test_input
        )

        dss_result = result[
            "DSS Result"
        ]

        decision = dss_result[
            "Decision"
        ]

        # If execution succeeded:
        if expected == "PASS_EXECUTION":

            test_status = "PASS"

            observed = (
                "Executed successfully"
            )

        else:

            # An invalid case executed when we expected rejection.
            test_status = "REVIEW"

            observed = (
                "Input was accepted by backend"
            )


        results.append({
            "Test":
                case["Test"],

            "Expected Behavior":
                expected,

            "Observed Behavior":
                observed,

            "Incident Type":
                decision.get(
                    "Primary Type",
                    ""
                ),

            "Arrest Probability (%)":
                decision.get(
                    "Arrest Probability (%)",
                    ""
                ),

            "Result":
                test_status,

            "Error / Note":
                ""
        })


    except Exception as exc:

        if expected == "SAFE_REJECTION":

            test_status = "PASS"

            observed = (
                "Rejected safely"
            )

        else:

            test_status = "FAIL"

            observed = (
                "Unexpected exception"
            )


        results.append({
            "Test":
                case["Test"],

            "Expected Behavior":
                expected,

            "Observed Behavior":
                observed,

            "Incident Type":
                test_input[
                    "primary_type"
                ],

            "Arrest Probability (%)":
                "",

            "Result":
                test_status,

            "Error / Note":
                str(exc)
        })


# ------------------------------------------------------------
# Results table
# ------------------------------------------------------------

boundary_df = pd.DataFrame(
    results
)

display(
    boundary_df
)


# ------------------------------------------------------------
# Summary
# ------------------------------------------------------------

pass_count = (
    boundary_df["Result"]
    == "PASS"
).sum()

review_count = (
    boundary_df["Result"]
    == "REVIEW"
).sum()

fail_count = (
    boundary_df["Result"]
    == "FAIL"
).sum()


print()
print("=" * 80)
print("PHASE B SUMMARY")
print("=" * 80)

print(
    f"PASS   : {pass_count}/{len(test_cases)}"
)

print(
    f"REVIEW : {review_count}/{len(test_cases)}"
)

print(
    f"FAIL   : {fail_count}/{len(test_cases)}"
)

print("-" * 80)


if (
    fail_count == 0
    and
    review_count == 0
):

    print(
        "FINAL RESULT : PASS — "
        "All tested boundaries and invalid inputs behaved as expected."
    )

elif fail_count == 0:

    print(
        "FINAL RESULT : REVIEW REQUIRED — "
        "No valid scenario failed, but one or more invalid values "
        "were accepted by the backend and should be assessed."
    )

else:

    print(
        "FINAL RESULT : FAIL — "
        "At least one valid boundary condition caused an unexpected failure."
    )

print("=" * 80)

ARMY PROVOST DSS — PHASE B BOUNDARY & ERROR TESTING


,Test,Expected Behavior,Observed Behavior,Incident Type,Arrest Probability (%),Result,Error / Note
0,Month lower boundary,PASS_EXECUTION,Executed successfully,BATTERY,38.2,PASS,
1,Month upper boundary,PASS_EXECUTION,Executed successfully,BATTERY,38.93,PASS,
2,Hour lower boundary,PASS_EXECUTION,Executed successfully,BATTERY,39.94,PASS,
3,Hour upper boundary,PASS_EXECUTION,Executed successfully,BATTERY,40.79,PASS,
4,Administrative zero boundary,PASS_EXECUTION,Executed successfully,BATTERY,38.23,PASS,
5,Domestic = True,PASS_EXECUTION,Executed successfully,BATTERY,36.89,PASS,
6,Invalid month = 13,SAFE_REJECTION,Rejected safely,BATTERY,,PASS,Month must be between 1 and 12.
7,Invalid month = 0,SAFE_REJECTION,Rejected safely,BATTERY,,PASS,Month must be between 1 and 12.
8,Invalid hour = 24,SAFE_REJECTION,Rejected safely,BATTERY,,PASS,Hour must be between 0 and 23.
9,Unknown incident type,SAFE_REJECTION,Rejected safely,INVALID INCIDENT TYPE,,PASS,'INVALID INCIDENT TYPE' is not present in the ...



PHASE B SUMMARY
PASS   : 10/10
REVIEW : 0/10
FAIL   : 0/10
--------------------------------------------------------------------------------
FINAL RESULT : PASS — All tested boundaries and invalid inputs behaved as expected.


In [18]:
# ============================================================
# PHASE C — FINAL REGRESSION VALIDATION
# Army Provost DSS Version 1.0
# ============================================================

import sys
import importlib

PROJECT_ROOT = "/content/drive/MyDrive/Army_Provost_ML_Project"
OUTPUTS_PATH = f"{PROJECT_ROOT}/Outputs"

if OUTPUTS_PATH not in sys.path:
    sys.path.insert(0, OUTPUTS_PATH)

import army_provost_dss_backend
importlib.reload(army_provost_dss_backend)

from army_provost_dss_backend import execute_dashboard_dss


# ------------------------------------------------------------
# Golden BATTERY regression input
# ------------------------------------------------------------

golden_input = {
    "primary_type": "BATTERY",
    "location_description": "STREET",
    "domestic": False,
    "year": 2025,
    "month": 7,
    "day": 15,
    "hour": 20,
    "district": 1,
    "beat": 100,
    "ward": 1,
    "community_area": 32
}


# ------------------------------------------------------------
# Execute DSS
# ------------------------------------------------------------

result = execute_dashboard_dss(
    **golden_input
)

validated_input = result[
    "Validated Input"
]

dss_result = result[
    "DSS Result"
]

decision = dss_result[
    "Decision"
]

record = dss_result[
    "Decision Record"
]

guidance = dss_result[
    "Response Guidance"
]


# ------------------------------------------------------------
# Regression checks
# ------------------------------------------------------------

checks = {

    "Incident Type = BATTERY":
        decision[
            "Primary Type"
        ]
        == "BATTERY",

    "Provost Category preserved":
        decision[
            "Provost Incident Category"
        ]
        == "Personnel & Physical Safety",

    "Subcategory preserved":
        decision[
            "Incident Subcategory"
        ]
        == "Physical Assault",

    "Priority = High":
        decision[
            "Priority"
        ]
        == "High",

    "Arrest Probability = 39.57%":
        abs(
            float(
                decision[
                    "Arrest Probability (%)"
                ]
            )
            - 39.57
        )
        <= 0.01,

    "ML Assessment preserved":
        decision[
            "Arrest Prediction"
        ]
        == "Less Likely Arrest",

    "Response Type preserved":
        decision[
            "Recommended Response Type"
        ]
        == "Personnel Safety / Provost Response",

    "Decision ID generated":
        bool(
            str(
                record.get(
                    "Decision ID",
                    ""
                )
            ).strip()
        ),

    "Decision Timestamp generated":
        bool(
            str(
                record.get(
                    "Decision Timestamp",
                    ""
                )
            ).strip()
        ),

    "Response Guidance available":
        all(
            bool(
                str(
                    guidance.get(
                        key,
                        ""
                    )
                ).strip()
            )
            for key in [
                "Immediate Operator Guidance",
                "Coordination / Notification",
                "Scene / Evidence Considerations",
                "Escalation / Follow-up"
            ]
        ),

    "Guidance Status preserved":
        guidance[
            "Guidance Status"
        ]
        == "Prototype Decision-Support Guidance"
}


# ------------------------------------------------------------
# Display results
# ------------------------------------------------------------

print("=" * 78)
print("ARMY PROVOST DSS — PHASE C FINAL REGRESSION VALIDATION")
print("=" * 78)

for check_name, passed in checks.items():

    print(
        f"{'PASS' if passed else 'FAIL':<6} "
        f"— {check_name}"
    )


print("-" * 78)

print(
    f"Arrest Probability       : "
    f"{decision['Arrest Probability (%)']:.2f}%"
)

print(
    f"ML Assessment            : "
    f"{decision['Arrest Prediction']}"
)

print(
    f"Priority                 : "
    f"{decision['Priority']}"
)

print(
    f"Recommended Response     : "
    f"{decision['Recommended Response Type']}"
)

print(
    f"Decision ID              : "
    f"{record['Decision ID']}"
)

print(
    f"Decision Timestamp       : "
    f"{record['Decision Timestamp']}"
)

print("-" * 78)


all_passed = all(
    checks.values()
)

if all_passed:

    print(
        "FINAL RESULT : PASS — "
        "Golden regression and core DSS invariants are preserved."
    )

else:

    print(
        "FINAL RESULT : FAIL — "
        "One or more regression invariants changed."
    )

print("=" * 78)

ARMY PROVOST DSS — PHASE C FINAL REGRESSION VALIDATION
PASS   — Incident Type = BATTERY
PASS   — Provost Category preserved
PASS   — Subcategory preserved
PASS   — Priority = High
PASS   — Arrest Probability = 39.57%
PASS   — ML Assessment preserved
PASS   — Response Type preserved
PASS   — Decision ID generated
PASS   — Decision Timestamp generated
PASS   — Response Guidance available
PASS   — Guidance Status preserved
------------------------------------------------------------------------------
Arrest Probability       : 39.57%
ML Assessment            : Less Likely Arrest
Priority                 : High
Recommended Response     : Personnel Safety / Provost Response
Decision ID              : AP-DSS-20260809-172001-782818
Decision Timestamp       : 2026-08-09T17:20:01
------------------------------------------------------------------------------
FINAL RESULT : PASS — Golden regression and core DSS invariants are preserved.
